# Imports

In [84]:
import os
import sqlite3

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import lightgbm as lgb
import optuna
import shap

import umap
import hdbscan

from joblib import dump, load
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import classification_report, recall_score

# Constants

In [85]:
RANDOM_SEED = 42

# Inputs

In [86]:
DB_PATH = "../data/sqlite/data.db"
TABLE   = "network_data"

# Samples per class (defined by the `label` column).
# - int  → same cap for every class
# - None → all available data (careful: may crash on large datasets)
SAMPLES_PER_CLASS: int | None = 10_000
# Per-class overrides — take precedence over SAMPLES_PER_CLASS.
SAMPLES_PER_CLASS_OVERRIDE: dict = {}  # e.g. {"benign": 20_000}

# Binary sampling: (n_benign, n_attack) — overrides SAMPLES_PER_CLASS when set.
# Loads n_benign rows from BENIGN_LABEL and n_attack rows from all other classes combined.
# Example: (50_000, 50_000) → 50k benign + 50k attack (pooled from all attack classes)
BINARY_SAMPLING: "tuple[int, int] | None" = None
BINARY_SAMPLING = [400000, 400000]

BENIGN_LABEL: str = "benign"

# Data

## Load Data

In [87]:
def load_dataset_from_db(
    db_path: str,
    table: str,
    samples_per_class: "int | None" = None,
    override: dict = {},
    binary_sampling: "tuple[int, int] | None" = None,
    benign_label: str = "benign",
) -> pd.DataFrame:
    """
    Load data from SQLite, sampling per class directly in the DB so that
    only the selected rows are pulled into memory.

    SQLite handles sampling with ORDER BY RANDOM() LIMIT n, avoiding the
    need to load the full dataset into pandas before filtering.

    When binary_sampling=(n_benign, n_attack) is provided, samples n_benign
    rows from benign_label and distributes n_attack evenly across all attack
    classes (n_attack // num_attack_classes per class), ignoring
    samples_per_class and override.
    """
    conn = sqlite3.connect(db_path)

    if binary_sampling is not None:
        n_benign, n_attack = binary_sampling

        q_benign = f'SELECT * FROM "{table}" WHERE label = ? ORDER BY RANDOM() LIMIT ?'
        df_benign = pd.read_sql(q_benign, conn, params=(benign_label, n_benign))
        print(f"  {benign_label}: {len(df_benign):,} rows")

        attack_classes = pd.read_sql(
            f'SELECT DISTINCT label FROM "{table}" WHERE label != ? AND label IS NOT NULL',
            conn, params=(benign_label,)
        )["label"].tolist()

        n_per_class = n_attack // len(attack_classes)
        attack_frames = []
        for cls in sorted(attack_classes):
            q_cls = f'SELECT * FROM "{table}" WHERE label = ? ORDER BY RANDOM() LIMIT ?'
            df_cls = pd.read_sql(q_cls, conn, params=(cls, n_per_class))
            attack_frames.append(df_cls)
            print(f"  {cls}: {len(df_cls):,} rows")

        conn.close()
        result = pd.concat([df_benign] + attack_frames, ignore_index=True).sample(frac=1, random_state=42)
        print(f"\nTotal: {len(result):,} rows")
        return result

    classes = pd.read_sql(
        f'SELECT DISTINCT label FROM "{table}" WHERE label IS NOT NULL', conn
    )["label"].tolist()
    print(f"Classes: {sorted(classes)}")

    frames = []
    for cls in classes:
        n = override.get(cls, samples_per_class)

        if n is None:
            query = f'SELECT * FROM "{table}" WHERE label = ?'
            df_cls = pd.read_sql(query, conn, params=(cls,))
        else:
            query = f'SELECT * FROM "{table}" WHERE label = ? ORDER BY RANDOM() LIMIT ?'
            df_cls = pd.read_sql(query, conn, params=(cls, n))

        frames.append(df_cls)
        print(f"  {cls}: {len(df_cls):,} rows")

    conn.close()

    result = pd.concat(frames, ignore_index=True).sample(frac=1, random_state=42)
    print(f"\nTotal: {len(result):,} rows")
    return result

In [88]:
def load_dataset_from_csv(
    data_root: str,
    dataset: str,
    samples_per_class: "int | None" = None,
    override: dict = {},
    random_state: int = 42,
) -> pd.DataFrame:
    """
    Alternative: load from CSV files directly.
    Only use for small datasets — large ones will exhaust RAM.
    """
    import glob
    dataset_path = os.path.join(data_root, dataset)
    csv_paths = sorted(glob.glob(os.path.join(dataset_path, "**", "*.csv"), recursive=True))
    if not csv_paths:
        raise FileNotFoundError(f"No CSVs found in: {dataset_path}")

    df = pd.concat((pd.read_csv(p) for p in csv_paths), ignore_index=True)

    if samples_per_class is None and not override:
        return df

    sampled = []
    for cls in df["label"].unique():
        n = override.get(cls, samples_per_class)
        cls_df = df[df["label"] == cls]
        sampled.append(cls_df.sample(min(n, len(cls_df)), random_state=random_state) if n else cls_df)

    return pd.concat(sampled, ignore_index=True).sample(frac=1, random_state=random_state)

In [ ]:
df = load_dataset_from_db(
    DB_PATH,
    TABLE,
    samples_per_class=SAMPLES_PER_CLASS,
    override=SAMPLES_PER_CLASS_OVERRIDE,
    binary_sampling=BINARY_SAMPLING,
    benign_label=BENIGN_LABEL,
)
df.head()

  benign: 400,000 rows
  bruteforce: 16,397 rows


## Data Cleaning

### Removing useless features

Some columns were discarded because they did not provide useful information for training, or because they biased the information:
- Timestamp

In [ ]:
removed_features: dict[str, list[str]] = {}

print("Before:")
print(f"  Shape: {df.shape}")
print(f"  Columns: {list(df.columns)}")

USELESS_FEATURES = ["timestamp"]#, "src_ip", "dst_ip"]
actually_dropped = [c for c in USELESS_FEATURES if c in df.columns]
df = df.drop(columns=actually_dropped)
removed_features["useless_features"] = actually_dropped

print("\nAfter:")
print(f"  Shape: {df.shape}")
print(f"  Columns: {list(df.columns)}")

Before:
  Shape: (200000, 83)
  Columns: ['src_ip', 'dst_ip', 'src_port', 'dst_port', 'protocol', 'timestamp', 'flow_duration', 'flow_byts_s', 'flow_pkts_s', 'fwd_pkts_s', 'bwd_pkts_s', 'tot_fwd_pkts', 'tot_bwd_pkts', 'totlen_fwd_pkts', 'totlen_bwd_pkts', 'fwd_pkt_len_max', 'fwd_pkt_len_min', 'fwd_pkt_len_mean', 'fwd_pkt_len_std', 'bwd_pkt_len_max', 'bwd_pkt_len_min', 'bwd_pkt_len_mean', 'bwd_pkt_len_std', 'pkt_len_max', 'pkt_len_min', 'pkt_len_mean', 'pkt_len_std', 'pkt_len_var', 'fwd_header_len', 'bwd_header_len', 'fwd_seg_size_min', 'fwd_act_data_pkts', 'flow_iat_mean', 'flow_iat_max', 'flow_iat_min', 'flow_iat_std', 'fwd_iat_tot', 'fwd_iat_max', 'fwd_iat_min', 'fwd_iat_mean', 'fwd_iat_std', 'bwd_iat_tot', 'bwd_iat_max', 'bwd_iat_min', 'bwd_iat_mean', 'bwd_iat_std', 'fwd_psh_flags', 'bwd_psh_flags', 'fwd_urg_flags', 'bwd_urg_flags', 'fin_flag_cnt', 'syn_flag_cnt', 'rst_flag_cnt', 'psh_flag_cnt', 'ack_flag_cnt', 'urg_flag_cnt', 'ece_flag_cnt', 'down_up_ratio', 'pkt_size_avg', 'fwd_se

### Removing spaces from feature names

In [ ]:
spaces_before = [c for c in df.columns if c != c.strip()]
print("Before - columns with spaces:", spaces_before)

df.columns = df.columns.str.strip()

spaces_after = [c for c in df.columns if c != c.strip()]
print("After  - columns with spaces:", spaces_after)
df.head()

Before - columns with spaces: []
After  - columns with spaces: []


,src_ip,dst_ip,src_port,dst_port,protocol,flow_duration,flow_byts_s,flow_pkts_s,fwd_pkts_s,bwd_pkts_s,...,active_max,active_min,active_mean,active_std,idle_max,idle_min,idle_mean,idle_std,cwr_flag_count,label
119737,192.168.137.196,192.168.137.131,32953,554,6,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,ddos
72272,192.168.137.47,192.168.137.1,59691,53,17,0.194300,2120.433016,10.293364,5.146682,5.146682,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,benign
158154,192.168.137.11,52.88.201.206,49078,8883,6,186.647029,4.671920,0.064292,0.042862,0.021431,...,0.089099,0.088412,0.088811,0.000291,62.101998,62.082369,62.091401,0.008089,0,mitm
65426,13.225.196.108,192.168.137.250,443,35730,6,0.007991,29909.256952,500.573338,125.143335,375.430004,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,benign
30074,192.168.1.1,224.0.0.251,38062,5353,17,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,benign


### Removing rows with inoperative values

In [ ]:
numeric_df = df.select_dtypes(include="number")

print("Before:")
print(f"  Shape: {df.shape}")
print(f"  NaN count:  {df.isna().sum().sum()}")
print(f"  Inf count:  {np.isinf(numeric_df).sum().sum()}")

df = df.replace([np.inf, -np.inf], np.nan).dropna()

numeric_df = df.select_dtypes(include="number")
print("\nAfter:")
print(f"  Shape: {df.shape}")
print(f"  NaN count:  {df.isna().sum().sum()}")
print(f"  Inf count:  {np.isinf(numeric_df).sum().sum()}")
df.head()

Before:
  Shape: (200000, 82)
  NaN count:  0
  Inf count:  0

After:
  Shape: (200000, 82)
  NaN count:  0
  Inf count:  0


,src_ip,dst_ip,src_port,dst_port,protocol,flow_duration,flow_byts_s,flow_pkts_s,fwd_pkts_s,bwd_pkts_s,...,active_max,active_min,active_mean,active_std,idle_max,idle_min,idle_mean,idle_std,cwr_flag_count,label
119737,192.168.137.196,192.168.137.131,32953,554,6,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,ddos
72272,192.168.137.47,192.168.137.1,59691,53,17,0.194300,2120.433016,10.293364,5.146682,5.146682,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,benign
158154,192.168.137.11,52.88.201.206,49078,8883,6,186.647029,4.671920,0.064292,0.042862,0.021431,...,0.089099,0.088412,0.088811,0.000291,62.101998,62.082369,62.091401,0.008089,0,mitm
65426,13.225.196.108,192.168.137.250,443,35730,6,0.007991,29909.256952,500.573338,125.143335,375.430004,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,benign
30074,192.168.1.1,224.0.0.251,38062,5353,17,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,benign


### Removing duplicate rows

In [ ]:
print("Before:")
print(f"  Shape: {df.shape}")
print(f"  Duplicate rows: {df.duplicated().sum()}")

df = df.drop_duplicates()

print("\nAfter:")
print(f"  Shape: {df.shape}")
print(f"  Duplicate rows: {df.duplicated().sum()}")
df.head()

Before:
  Shape: (200000, 82)
  Duplicate rows: 2800

After:
  Shape: (197200, 82)
  Duplicate rows: 0


,src_ip,dst_ip,src_port,dst_port,protocol,flow_duration,flow_byts_s,flow_pkts_s,fwd_pkts_s,bwd_pkts_s,...,active_max,active_min,active_mean,active_std,idle_max,idle_min,idle_mean,idle_std,cwr_flag_count,label
119737,192.168.137.196,192.168.137.131,32953,554,6,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,ddos
72272,192.168.137.47,192.168.137.1,59691,53,17,0.194300,2120.433016,10.293364,5.146682,5.146682,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,benign
158154,192.168.137.11,52.88.201.206,49078,8883,6,186.647029,4.671920,0.064292,0.042862,0.021431,...,0.089099,0.088412,0.088811,0.000291,62.101998,62.082369,62.091401,0.008089,0,mitm
65426,13.225.196.108,192.168.137.250,443,35730,6,0.007991,29909.256952,500.573338,125.143335,375.430004,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,benign
30074,192.168.1.1,224.0.0.251,38062,5353,17,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,benign


### Removing highly correlated features

In [ ]:
CORR_THRESHOLD = 0.95

numeric_cols = df.select_dtypes(include="number").columns.tolist()
print("Before:")
print(f"  Shape: {df.shape}")

corr_matrix = df[numeric_cols].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > CORR_THRESHOLD)]
print(f"  Dropped (corr > {CORR_THRESHOLD}): {to_drop}")

df = df.drop(columns=to_drop)
removed_features["high_correlation"] = to_drop

print("\nAfter:")
print(f"  Shape: {df.shape}")
df.head()

Before:
  Shape: (197200, 82)
  Dropped (corr > 0.95): ['fwd_seg_size_min', 'fwd_act_data_pkts', 'fwd_iat_tot', 'fwd_iat_max', 'fwd_psh_flags', 'psh_flag_cnt', 'ack_flag_cnt', 'urg_flag_cnt', 'pkt_size_avg', 'fwd_seg_size_avg', 'bwd_seg_size_avg', 'fwd_pkts_b_avg', 'bwd_pkts_b_avg', 'subflow_bwd_pkts', 'active_mean', 'active_std', 'idle_max', 'idle_mean']

After:
  Shape: (197200, 64)


,src_ip,dst_ip,src_port,dst_port,protocol,flow_duration,flow_byts_s,flow_pkts_s,fwd_pkts_s,bwd_pkts_s,...,subflow_fwd_byts,subflow_bwd_byts,init_fwd_win_byts,init_bwd_win_byts,active_max,active_min,idle_min,idle_std,cwr_flag_count,label
119737,192.168.137.196,192.168.137.131,32953,554,6,0.000000,0.000000,0.000000,0.000000,0.000000,...,1500,0,512,-1,0.000000,0.000000,0.000000,0.000000,0,ddos
72272,192.168.137.47,192.168.137.1,59691,53,17,0.194300,2120.433016,10.293364,5.146682,5.146682,...,69,343,-1,-1,0.000000,0.000000,0.000000,0.000000,0,benign
158154,192.168.137.11,52.88.201.206,49078,8883,6,186.647029,4.671920,0.064292,0.042862,0.021431,...,135,83,1456,123,0.089099,0.088412,62.082369,0.008089,0,mitm
65426,13.225.196.108,192.168.137.250,443,35730,6,0.007991,29909.256952,500.573338,125.143335,375.430004,...,52,187,135,1448,0.000000,0.000000,0.000000,0.000000,0,benign
30074,192.168.1.1,224.0.0.251,38062,5353,17,0.000000,0.000000,0.000000,0.000000,0.000000,...,71,0,-1,-1,0.000000,0.000000,0.000000,0.000000,0,benign


### Removing near-zero variance features

In [ ]:
VARIANCE_THRESHOLD = 1e-4

numeric_cols = df.select_dtypes(include="number").columns.tolist()
print("Before:")
print(f"  Shape: {df.shape}")

variances = df[numeric_cols].var()
low_var_cols = variances[variances < VARIANCE_THRESHOLD].index.tolist()
print(f"  Dropped (var < {VARIANCE_THRESHOLD}): {low_var_cols}")

df = df.drop(columns=low_var_cols)
removed_features["near_zero_variance"] = low_var_cols

print("\nAfter:")
print(f"  Shape: {df.shape}")
df.head()

Before:
  Shape: (197200, 64)
  Dropped (var < 0.0001): ['fwd_urg_flags', 'bwd_urg_flags']

After:
  Shape: (197200, 62)


,src_ip,dst_ip,src_port,dst_port,protocol,flow_duration,flow_byts_s,flow_pkts_s,fwd_pkts_s,bwd_pkts_s,...,subflow_fwd_byts,subflow_bwd_byts,init_fwd_win_byts,init_bwd_win_byts,active_max,active_min,idle_min,idle_std,cwr_flag_count,label
119737,192.168.137.196,192.168.137.131,32953,554,6,0.000000,0.000000,0.000000,0.000000,0.000000,...,1500,0,512,-1,0.000000,0.000000,0.000000,0.000000,0,ddos
72272,192.168.137.47,192.168.137.1,59691,53,17,0.194300,2120.433016,10.293364,5.146682,5.146682,...,69,343,-1,-1,0.000000,0.000000,0.000000,0.000000,0,benign
158154,192.168.137.11,52.88.201.206,49078,8883,6,186.647029,4.671920,0.064292,0.042862,0.021431,...,135,83,1456,123,0.089099,0.088412,62.082369,0.008089,0,mitm
65426,13.225.196.108,192.168.137.250,443,35730,6,0.007991,29909.256952,500.573338,125.143335,375.430004,...,52,187,135,1448,0.000000,0.000000,0.000000,0.000000,0,benign
30074,192.168.1.1,224.0.0.251,38062,5353,17,0.000000,0.000000,0.000000,0.000000,0.000000,...,71,0,-1,-1,0.000000,0.000000,0.000000,0.000000,0,benign


In [ ]:
final_features = df.columns.tolist()
all_removed    = [col for cols in removed_features.values() for col in cols]

sep  = "=" * 60
sep2 = "-" * 60
lines = [
    sep,
    "FEATURE REPORT — DATA CLEANING SUMMARY",
    sep,
    "",
    f"Final features used ({len(final_features)}):",
]
for feat in sorted(final_features):
    lines.append(f"  + {feat}")

lines += ["", f"Excluded features ({len(all_removed)}):"]
for reason, cols in removed_features.items():
    if cols:
        lines += ["", f"  [{reason}]"]
        for col in sorted(cols):
            lines.append(f"    - {col}")

report = "\n".join(lines) + "\n"
print(report)

os.makedirs("../docs", exist_ok=True)
with open("../docs/features_report.txt", "w") as fh:
    fh.write(report)

print("Saved → ../docs/features_report.txt")

FEATURE REPORT — DATA CLEANING SUMMARY

Final features used (62):
  + active_max
  + active_min
  + bwd_blk_rate_avg
  + bwd_byts_b_avg
  + bwd_header_len
  + bwd_iat_max
  + bwd_iat_mean
  + bwd_iat_min
  + bwd_iat_std
  + bwd_iat_tot
  + bwd_pkt_len_max
  + bwd_pkt_len_mean
  + bwd_pkt_len_min
  + bwd_pkt_len_std
  + bwd_pkts_s
  + bwd_psh_flags
  + cwr_flag_count
  + down_up_ratio
  + dst_ip
  + dst_port
  + ece_flag_cnt
  + fin_flag_cnt
  + flow_byts_s
  + flow_duration
  + flow_iat_max
  + flow_iat_mean
  + flow_iat_min
  + flow_iat_std
  + flow_pkts_s
  + fwd_blk_rate_avg
  + fwd_byts_b_avg
  + fwd_header_len
  + fwd_iat_mean
  + fwd_iat_min
  + fwd_iat_std
  + fwd_pkt_len_max
  + fwd_pkt_len_mean
  + fwd_pkt_len_min
  + fwd_pkt_len_std
  + fwd_pkts_s
  + idle_min
  + idle_std
  + init_bwd_win_byts
  + init_fwd_win_byts
  + label
  + pkt_len_max
  + pkt_len_mean
  + pkt_len_min
  + pkt_len_std
  + pkt_len_var
  + protocol
  + rst_flag_cnt
  + src_ip
  + src_port
  + subflow_bwd_b

# Phase 1 — Binary Classification (Benign vs Malicious)

In [ ]:
y = df["label"].apply(lambda x: "benign" if x == "benign" else "attack")
X = df.drop(columns=df.select_dtypes(exclude="number").columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

A prioridade aqui é o **recall**, para minimizar falsos negativos — é preferível suspeitar de tráfego benigno do que deixar um ataque passar como benigno.

In [ ]:
def objective(trial):
    threshold = trial.suggest_float("threshold", 0.01, 0.5)

    params = {
        "objective": "multiclass",
        "num_class": y_train.nunique(),
        "metric": "multi_logloss",
        "boosting_type": "gbdt",
        "random_state": RANDOM_SEED,
        "num_leaves": trial.suggest_int("num_leaves", 31, 255),
        "max_depth": trial.suggest_int("max_depth", -1, 16),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
        "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 1.0),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 200, 800),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "subsample_freq": trial.suggest_int("subsample_freq", 1, 7),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "max_bin": trial.suggest_int("max_bin", 128, 512),
        "class_weight": "balanced",
        "verbosity": -1,
        "force_row_wise": True,
        "n_jobs": 3,
    }

    model = lgb.LGBMClassifier(**params)
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)

    scores = []
    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model.fit(X_tr, y_tr)
        probs = model.predict_proba(X_val)

        attack_idx = list(model.classes_).index("attack")
        y_pred = (probs[:, attack_idx] > threshold).astype(int)
        y_val_bin = (y_val == "attack").astype(int)

        scores.append(recall_score(y_val_bin, y_pred))

    return np.mean(scores)

In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20, timeout=1800, show_progress_bar=True)

[I 2026-05-05 15:18:17,907] A new study created in memory with name: no-name-4e7d91a4-76e2-4fe4-92c4-93d76dc3fec4
Best trial: 0. Best value: 0.881641:   5%|▌         | 1/20 [00:18<05:49, 18.37s/it, 18.37/1800 seconds]

[I 2026-05-05 15:18:36,277] Trial 0 finished with value: 0.8816407710520638 and parameters: {'threshold': 0.29765544537057764, 'num_leaves': 142, 'max_depth': 5, 'min_child_samples': 83, 'min_split_gain': 0.1091749548741544, 'learning_rate': 0.008122749650651694, 'n_estimators': 636, 'subsample': 0.9236650393649685, 'subsample_freq': 4, 'colsample_bytree': 0.8470165708083288, 'reg_alpha': 2.0993758664202448e-05, 'reg_lambda': 0.002100727190724079, 'max_bin': 400}. Best is trial 0 with value: 0.8816407710520638.


Best trial: 1. Best value: 0.921195:  10%|█         | 2/20 [01:28<14:36, 48.72s/it, 88.33/1800 seconds]

[I 2026-05-05 15:19:46,242] Trial 1 finished with value: 0.9211949597598309 and parameters: {'threshold': 0.2410750940823611, 'num_leaves': 200, 'max_depth': 0, 'min_child_samples': 22, 'min_split_gain': 0.23000875936006904, 'learning_rate': 0.008724305663513903, 'n_estimators': 740, 'subsample': 0.7829193736890137, 'subsample_freq': 1, 'colsample_bytree': 0.8373614559884138, 'reg_alpha': 0.016489919730255962, 'reg_lambda': 1.548087303854124e-05, 'max_bin': 511}. Best is trial 1 with value: 0.9211949597598309.


Best trial: 1. Best value: 0.921195:  15%|█▌        | 3/20 [02:06<12:24, 43.82s/it, 126.33/1800 seconds]

...

[I 2026-05-05 15:24:10,748] Trial 17 finished with value: 0.8479376078326001 and parameters: {'threshold': 0.36522204675560493, 'num_leaves': 133, 'max_depth': 7, 'min_child_samples': 19, 'min_split_gain': 0.8105443714310978, 'learning_rate': 0.005061796645875768, 'n_estimators': 438, 'subsample': 0.9219036680898274, 'subsample_freq': 2, 'colsample_bytree': 0.8884755065898718, 'reg_alpha': 0.00029851869483580435, 'reg_lambda': 0.23445837691967, 'max_bin': 215}. Best is trial 11 with value: 1.0.


Best trial: 11. Best value: 1:  95%|█████████▌| 19/20 [06:02<00:13, 13.73s/it, 362.44/1800 seconds]

[I 2026-05-05 15:24:20,344] Trial 18 finished with value: 0.942475866290315 and parameters: {'threshold': 0.22804622056617768, 'num_leaves': 183, 'max_depth': 7, 'min_child_samples': 39, 'min_split_gain': 0.883579866064268, 'learning_rate': 0.036992565883988984, 'n_estimators': 277, 'subsample': 0.8617826382688032, 'subsample_freq': 5, 'colsample_bytree': 0.9607765146878279, 'reg_alpha': 0.07890680971639583, 'reg_lambda': 4.438078026922994, 'max_bin': 291}. Best is trial 11 with value: 1.0.


Best trial: 11. Best value: 1: 100%|██████████| 20/20 [06:10<00:00, 18.52s/it, 370.35/1800 seconds]

[I 2026-05-05 15:24:28,259] Trial 19 finished with value: 0.9922408158508063 and parameters: {'threshold': 0.1001526985849013, 'num_leaves': 82, 'max_depth': 2, 'min_child_samples': 62, 'min_split_gain': 0.9903764536709821, 'learning_rate': 0.06726450652758638, 'n_estimators': 455, 'subsample': 0.9551274921609203, 'subsample_freq': 3, 'colsample_bytree': 0.8584927651830996, 'reg_alpha': 0.8072303177267004, 'reg_lambda': 0.0004918534171941377, 'max_bin': 379}. Best is trial 11 with value: 1.0.


In [ ]:
print("Best recall:", study.best_value)
print("Best params:", study.best_params)

Best recall: 1.0
Best params: {'threshold': 0.011843757585257204, 'num_leaves': 99, 'max_depth': 8, 'min_child_samples': 11, 'min_split_gain': 0.684257493180811, 'learning_rate': 0.05302438282536019, 'n_estimators': 379, 'subsample': 0.9939443449057954, 'subsample_freq': 3, 'colsample_bytree': 0.9172424827670729, 'reg_alpha': 3.0567189482761536, 'reg_lambda': 1.623303605138651, 'max_bin': 299}


In [ ]:
best = study.best_params.copy()
threshold = best.pop("threshold")

model = lgb.LGBMClassifier(**best)

In [ ]:
model.fit(X_train, y_train)

,boosting_type,'gbdt'
,num_leaves,99
,max_depth,8
,learning_rate,0.05302438282536019
,n_estimators,379
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.684257493180811
,min_child_weight,0.001
,min_child_samples,11


In [ ]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

      attack       0.95      0.84      0.89     19783
      benign       0.86      0.96      0.91     19657

    accuracy                           0.90     39440
   macro avg       0.91      0.90      0.90     39440
weighted avg       0.91      0.90      0.90     39440



In [ ]:
recall_score(y_test, y_pred, pos_label="benign")

0.9586406877956962

In [ ]:
dump(model, "../models/binary_classifier.pkl")

['../models/binary_classifier.pkl']

# Phase 2 — Multiclassification

# Phase 3 — Clustering